> ## ⚠ Update (2026-06): per-tier demand curves supersede the synthetic batch generator
>
> The batch-demand input to the environment no longer comes from the distribution-fitted synthetic generator (Datasets 5–6 below). Validation showed the generator's single-step job pulses made the aggregate **far burstier than the real trace** (synthetic batch curve peak/mean ≈ 195 vs ≈ 1.24 for the measured cell aggregate — see `thesis_overview.md` §7.10).
>
> **The environment now consumes real per-tier curves** (`data/cells/cell_{x}_tiers.csv`: non-deferrable service vs deferrable no-SLO batch, with `service + batch = measured aggregate`):
> - **Ground truth**: run the small standalone notebook **`extract_tier_curves.ipynb`** (splits `instance_usage` by priority tier; no need to re-run this notebook).
> - **Local approximation** (no Colab needed): `scripts/refit_freebeb_local.py` + `scripts/derive_tier_curves.py` derive the split from the full `jobs_*.csv` extracts.
>
> Datasets 1–4 below remain the source for the aggregate curves, power model, machines, and job metadata. Datasets 5–6 (distribution fits / generator params) are retained for **sensitivity experiments** (controlled load-intensity variation à la Grange et al. 2018, "Green IT scheduling for data center powered with renewable energy") and for the deadline model's `mean_duration`, but are no longer the primary batch-demand path.

# Comprehensive Google ClusterData2019 + PowerData2019 Extraction

**Purpose:** Extract all datasets needed to benchmark a convex-optimization-based 
green data center scheduler against three papers from the Lin et al. (2024) survey:

| Paper | Key Result | What We Extract |
|-------|-----------|-----------------|
| **Grange et al. (2018)** — Batch scheduling w/ renewable awareness | 49% brown-energy ↓, 51% cost ↓ | Batch job statistical profiles (distributions for workload generator) |
| **Xu et al. (2020)** — Self-adaptive brownout + batch deferral | 21% brown-energy ↓, 10% renewable ↑ | Mixed batch/service classification, aggregate utilization curves |
| **Haghshenas et al. (2022)** — Infrastructure-aware heterogeneous scheduling | Energy-cost minimization | Heterogeneous machine fleet, cooling-relevant utilization, workload mix |

## Datasets Produced

| # | Dataset | File(s) | Size | Used By |
|---|---------|---------|------|---------|
| 1 | Cell CPU utilization (5-min) | `cells/cell_{a..d}.csv` | ~50K rows/cell | All three |
| 2 | Power model (CPU→Power) | `power_model_params.json`, `power_model_scatter.csv` | <1 MB | All three |
| 3 | Machine attributes (heterogeneous fleet) | `machines/machines_{a..d}.csv`, `machines_all.csv` | ~100K rows total | Haghshenas |
| 4 | Job metadata + batch/service classification | `jobs/jobs_{a..d}.csv` | ~1M rows/cell | Xu, Haghshenas |
| 5 | **Batch job statistical profiles** | `jobs/batch_distributions_{a..d}.json` | <100 KB | Grange |
| 6 | **Workload generator parameters** | `workload_generator_params.json` | <10 KB | Grange |

## Methodology Note

Following Grange et al.'s approach (which uses the Da Costa et al. 2016 workload generator), 
we do **not** replay raw instance events. Instead we:
1. Fit statistical distributions to real job characteristics (inter-arrival times, durations, resource requests)
2. Export the fitted parameters for use in a synthetic workload generator
3. Validate the fits with KS-test statistics

This is standard practice in the scheduling literature and produces a stronger experimental 
design than raw replay because it allows controlled variation of load intensity and job mix.

**Before running:** GCP project with BigQuery API enabled → https://console.cloud.google.com/flows/enableapi?apiid=bigquery

In [ ]:
# ============================================================
# Step 1: Authenticate and configure
# ============================================================
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'YOUR_PROJECT_ID_HERE'  # <-- CHANGE THIS

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)

# Quick test
test_q = """
SELECT SUM(cpu_cap) AS cpu_capacity
FROM (
    SELECT machine_id, MAX(capacity.cpus) AS cpu_cap
    FROM `google.com:google-cluster-data`.clusterdata_2019_a.machine_events
    GROUP BY 1
)
"""
result = client.query(test_q).to_dataframe()
print(f"Cell 'a' CPU capacity: {result['cpu_capacity'].iloc[0]:.2f}")
print(f'Authenticated with project: {PROJECT_ID}  ✓')

In [ ]:
import pandas as pd
import numpy as np
import json
import os
from scipy import stats

os.makedirs('data/cells', exist_ok=True)
os.makedirs('data/jobs', exist_ok=True)
os.makedirs('data/machines', exist_ok=True)

CELLS = ['a', 'b', 'c', 'd']

def get_cell_capacity(cell):
    """Get total CPU capacity for a cell."""
    query = f"""
    SELECT SUM(cpu_cap) AS cpu_capacity
    FROM (
        SELECT machine_id, MAX(capacity.cpus) AS cpu_cap
        FROM `google.com:google-cluster-data`.clusterdata_2019_{cell}.machine_events
        GROUP BY 1
    )
    """
    return float(client.query(query).to_dataframe()['cpu_capacity'].iloc[0])

# Pre-fetch capacities (reused by multiple datasets)
cell_capacities = {}
for cell in CELLS:
    cell_capacities[cell] = get_cell_capacity(cell)
    print(f"Cell {cell} CPU capacity: {cell_capacities[cell]:.2f}")

---
## Dataset 1: Cell-Level CPU Utilization (5-min intervals)

Aggregate normalized CPU demand per cell at 5-minute resolution.  
Used as the **demand curve input** for all three benchmark papers' optimization models.

In [ ]:
for cell in CELLS:
    print(f'\n--- Cell {cell}: CPU Utilization ---')
    cap = cell_capacities[cell]

    query = f"""
    SELECT
        CAST(FLOOR(start_time / (1e6 * 300)) AS INT64) AS time_bucket,
        SUM(average_usage.cpus) / {cap} AS cpu_demand_norm
    FROM `google.com:google-cluster-data`.clusterdata_2019_{cell}.instance_usage
    WHERE (alloc_collection_id IS NULL OR alloc_collection_id = 0)
        AND (end_time - start_time) >= (5 * 60 * 1e6)
    GROUP BY 1
    ORDER BY 1
    """
    df = client.query(query).to_dataframe()
    df['timestep'] = df['time_bucket'] - df['time_bucket'].min()
    df = df[['timestep', 'cpu_demand_norm']].copy()
    df['cpu_demand_norm'] = df['cpu_demand_norm'].clip(0.0, 1.0)

    path = f'data/cells/cell_{cell}.csv'
    df.to_csv(path, index=False)
    print(f'  {len(df)} rows → {path}')
    print(f'  Mean={df["cpu_demand_norm"].mean():.4f}, Max={df["cpu_demand_norm"].max():.4f}')

print('\n✅ Cell utilization done!')

---
## Dataset 2: Power Model (CPU → Power)

Fits a linear model `P = P_idle + slope × cpu_util` from Google PowerData2019.  
This is the standard server power model used across the sustainable DC literature.

In [ ]:
print('Extracting hourly power utilization...')
power_query = """
SELECT
    cell,
    CAST(FLOOR(time / (1e6 * 60 * 60)) AS INT64) AS hour_index,
    AVG(measured_power_util) AS avg_power_util
FROM `google.com:google-cluster-data`.`powerdata_2019.cell*`
WHERE NOT bad_measurement_data
    AND cell IN ('a', 'b', 'c', 'd')
GROUP BY 1, 2
ORDER BY 1, 2
"""
power_df = client.query(power_query).to_dataframe()
print(f'  {len(power_df)} power rows')


def get_cell_mem_capacity(cell):
    """Total (trace-normalized) memory capacity for a cell."""
    q = f"""
    SELECT SUM(mem_cap) AS mem_capacity FROM (
        SELECT machine_id, MAX(capacity.memory) AS mem_cap
        FROM `google.com:google-cluster-data`.clusterdata_2019_{cell}.machine_events
        GROUP BY 1
    )
    """
    return float(client.query(q).to_dataframe()['mem_capacity'].iloc[0])


# Hourly CPU *and* memory utilization per cell (memory is a candidate 2nd predictor)
all_cpu = []
for cell in CELLS:
    print(f'  Hourly CPU+memory for cell {cell}...')
    cap = cell_capacities[cell]
    memcap = get_cell_mem_capacity(cell)
    q = f"""
    SELECT
        CAST(FLOOR(start_time / (1e6 * 60 * 60)) AS INT64) AS hour_index,
        SUM(average_usage.cpus) / (12 * {cap}) AS avg_cpu_util,
        SUM(average_usage.memory) / (12 * {memcap}) AS avg_mem_util
    FROM `google.com:google-cluster-data`.clusterdata_2019_{cell}.instance_usage
    WHERE (alloc_collection_id IS NULL OR alloc_collection_id = 0)
        AND (end_time - start_time) >= (5 * 60 * 1e6)
    GROUP BY 1
    ORDER BY 1
    """
    cpu_df = client.query(q).to_dataframe()
    cpu_df['cell'] = cell
    all_cpu.append(cpu_df)

cpu_combined = pd.concat(all_cpu, ignore_index=True)
merged = pd.merge(cpu_combined, power_df, on=['cell', 'hour_index'], how='inner')
valid = (
    np.isfinite(merged['avg_cpu_util'])
    & np.isfinite(merged['avg_mem_util'])
    & np.isfinite(merged['avg_power_util'])
)
merged = merged[valid].copy()

cpu = merged['avg_cpu_util'].values
mem = merged['avg_mem_util'].values
pw = merged['avg_power_util'].values


def _fit(X, y):
    """Least-squares fit with intercept. X: (n, k) predictors. Returns (coefs, R2),
    coefs[0] = intercept."""
    A = np.column_stack([np.ones(len(y)), X])
    coef = np.linalg.lstsq(A, y, rcond=None)[0]
    pred = A @ coef
    ss_res = np.sum((y - pred) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    return coef, float(1.0 - ss_res / ss_tot)


# 1) Pooled CPU-only — this is what the env's PowerModel consumes (idle/slope/peak).
(intercept, slope), r2_cpu = _fit(cpu.reshape(-1, 1), pw)
intercept, slope = float(intercept), float(slope)

# 2) Pooled CPU + memory (multiple regression) — does memory absorb residual scatter?
coef_cm, r2_cm = _fit(np.column_stack([cpu, mem]), pw)

# 3) Per-cell CPU-only — each cell has a different machine mix → its own idle/slope.
per_cell = {}
for cell in CELLS:
    mk = merged['cell'].values == cell
    if mk.sum() > 10:
        (a0, a1), r2c = _fit(cpu[mk].reshape(-1, 1), pw[mk])
        per_cell[cell] = {
            'idle_power': float(a0), 'slope': float(a1),
            'peak_power': float(a0 + a1), 'r_squared': r2c, 'n': int(mk.sum()),
        }

# 4) Binned diagnostic — is the *mean* relationship linear? (separates model quality
#    from point-level noise / the narrow CPU-utilization range of aggregate load).
bins = np.linspace(cpu.min(), cpu.max(), 21)
bi = np.digitize(cpu, bins)
bx, by = [], []
for b in range(1, 21):
    mm = bi == b
    if mm.sum() > 5:
        bx.append(cpu[mm].mean())
        by.append(pw[mm].mean())
bx, by = np.array(bx), np.array(by)
_, r2_binned = _fit(bx.reshape(-1, 1), by)

params = {
    # --- consumed by env.power_model.PowerModel (unchanged schema) ---
    'idle_power': intercept,
    'peak_power': intercept + slope,
    'slope': slope,
    'r_squared': float(r2_cpu),
    'description': 'Linear power model: P = idle_power + slope * cpu_utilization',
    # --- richer diagnostics (informational; not read by the env) ---
    'cpu_util_range': [float(cpu.min()), float(cpu.max())],
    'binned_r_squared': float(r2_binned),
    'cpu_mem_model': {
        'idle_power': float(coef_cm[0]),
        'cpu_coef': float(coef_cm[1]),
        'mem_coef': float(coef_cm[2]),
        'r_squared': float(r2_cm),
        'description': 'P = idle_power + cpu_coef*cpu_util + mem_coef*mem_util',
    },
    'per_cell_cpu_model': per_cell,
}

print(f'\nPooled CPU-only:  idle={intercept:.4f} slope={slope:.4f}  R2={r2_cpu:.4f}')
print(f'Pooled CPU+mem:   R2={r2_cm:.4f}  (cpu={coef_cm[1]:.4f}, mem={coef_cm[2]:.4f})')
print(f'Binned-means R2:  {r2_binned:.4f}  <- central relationship; low point R2 is '
      f'scatter + narrow CPU range {params["cpu_util_range"]}')
for cell, pc in per_cell.items():
    print(f'  cell {cell}: idle={pc["idle_power"]:.3f} slope={pc["slope"]:.3f} '
          f'R2={pc["r_squared"]:.4f} (n={pc["n"]})')

with open('data/power_model_params.json', 'w') as f:
    json.dump(params, f, indent=2)
# Scatter now keeps cell + memory so the per-cell / CPU+mem fits can be reproduced
# locally without re-querying BigQuery.
pd.DataFrame({
    'cell': merged['cell'].values,
    'cpu_util': cpu,
    'mem_util': mem,
    'power_util': pw,
}).to_csv('data/power_model_scatter.csv', index=False)

print('✅ Power model done!')

---
## Dataset 3: Machine Attributes (Heterogeneous Fleet)

Per-machine CPU and memory capacity for modeling heterogeneous server infrastructure.  
**Haghshenas et al.** use this to model infrastructure-aware scheduling where different
server types have different power profiles and capabilities.

In [ ]:
all_machines = []
for cell in CELLS:
    print(f'\n--- Cell {cell}: Machines ---')
    query = f"""
    SELECT
        machine_id,
        MAX(capacity.cpus) AS cpu_capacity,
        MAX(capacity.memory) AS memory_capacity
    FROM `google.com:google-cluster-data`.clusterdata_2019_{cell}.machine_events
    GROUP BY 1
    HAVING cpu_capacity IS NOT NULL AND memory_capacity IS NOT NULL
    ORDER BY 1
    """
    df = client.query(query).to_dataframe()
    df['cell'] = cell
    df.to_csv(f'data/machines/machines_{cell}.csv', index=False)
    all_machines.append(df)

    print(f'  {len(df)} machines')
    print(f'  CPU  — unique types: {df["cpu_capacity"].nunique()}, '
          f'range: [{df["cpu_capacity"].min():.4f}, {df["cpu_capacity"].max():.4f}]')
    print(f'  Mem  — unique types: {df["memory_capacity"].nunique()}, '
          f'range: [{df["memory_capacity"].min():.4f}, {df["memory_capacity"].max():.4f}]')

combined = pd.concat(all_machines, ignore_index=True)
combined.to_csv('data/machines/machines_all.csv', index=False)
print(f'\n✅ {len(combined)} total machines across all cells')

---
## Dataset 4: Job Metadata + Batch/Service Classification

Job-level metadata from `collection_events` joined with aggregate resource requests
from `instance_events`. Each job is classified as **batch** or **service** using 
Google Borg conventions:

- `scheduling_class ≤ 1 AND priority < 200` → **Batch** (delay-tolerant)
- Otherwise → **Service** (latency-sensitive)

**Xu et al.** use this split for their brownout (service) + deferral (batch) algorithms.  
**Haghshenas et al.** use the heterogeneous workload mix for infrastructure-aware scheduling.

In [ ]:
for cell in CELLS:
    print(f'\n--- Cell {cell}: Job Metadata ---')

    # Batch vs service classification follows the Borg priority TIERS defined in
    # Tirmazi et al. (2020), "Borg: the Next Generation" (EuroSys '20), §2:
    #   free        : priority <= 99      (no charges, NO SLOs)          <-- deferrable
    #   best-effort  : priority 110-115    (queue-scheduled, NO SLOs)     <-- deferrable
    #   batch (beb)
    #   mid          : priority 116-119    (weak SLOs)
    #   production   : priority 120-359    (high availability; Borg EVICTS lower tiers to protect these)
    #   monitoring   : priority >= 360
    # Deferrable "batch" = the NO-SLO tiers (free OR beb). Strict beb alone is a
    # negligible share of these cells' load, so we use the union of the two
    # SLO-free tiers. Everything with an SLO (mid/production/monitoring) is
    # non-deferrable service. We classify by PRIORITY, not scheduling_class:
    # scheduling_class (latency-sensitivity 0-3) is orthogonal to tier, and
    # `scheduling_class <= 1` is dominated by latency-insensitive *production*
    # (priority 200) jobs, which are SLO-bound and must NOT be deferred.
    query = f"""
    WITH submits AS (
        SELECT
            collection_id,
            MIN(time) AS submit_time,
            MAX(scheduling_class) AS scheduling_class,
            MAX(priority) AS priority,
            CASE
                WHEN MAX(priority) <= 99 OR MAX(priority) BETWEEN 110 AND 115 THEN 'batch'
                ELSE 'service'
            END AS job_type
        FROM `google.com:google-cluster-data`.clusterdata_2019_{cell}.collection_events
        WHERE type = 0
        GROUP BY collection_id
    ),
    terminals AS (
        -- Use the LAST terminal event (MAX time): a job can be EVICT(4)ed and
        -- later FINISH(6), so MIN would truncate the runtime to the first eviction.
        -- MAX(type) prefers FINISH(6) over FAIL(5)/EVICT(4) as the recorded outcome.
        SELECT
            collection_id,
            MAX(time) AS end_time,
            MAX(type) AS terminal_type
        FROM `google.com:google-cluster-data`.clusterdata_2019_{cell}.collection_events
        WHERE type IN (4, 5, 6)
        GROUP BY collection_id
    ),
    resources AS (
        SELECT
            collection_id,
            COUNT(DISTINCT instance_index) AS num_tasks,
            AVG(resource_request.cpus) AS avg_cpu_request,
            AVG(resource_request.memory) AS avg_mem_request,
            SUM(resource_request.cpus) AS total_cpu_request,
            SUM(resource_request.memory) AS total_mem_request
        FROM `google.com:google-cluster-data`.clusterdata_2019_{cell}.instance_events
        WHERE type = 0
        GROUP BY collection_id
    )
    SELECT
        s.collection_id,
        s.submit_time,
        t.end_time,
        (t.end_time - s.submit_time) / 1e6 AS duration_sec,
        s.scheduling_class,
        s.priority,
        s.job_type,
        t.terminal_type,
        r.num_tasks,
        r.avg_cpu_request,
        r.avg_mem_request,
        r.total_cpu_request,
        r.total_mem_request
    FROM submits s
    LEFT JOIN terminals t ON s.collection_id = t.collection_id
    LEFT JOIN resources r ON s.collection_id = r.collection_id
    WHERE s.submit_time IS NOT NULL
    ORDER BY s.submit_time
    """
    df = client.query(query).to_dataframe()
    df.to_csv(f'data/jobs/jobs_{cell}.csv', index=False)

    n_batch = (df['job_type'] == 'batch').sum()
    n_service = (df['job_type'] == 'service').sum()
    print(f'  {len(df)} jobs — {n_batch} batch/no-SLO ({100*n_batch/len(df):.1f}%), '
          f'{n_service} service ({100*n_service/len(df):.1f}%)')

    # "Completed" = reached FINISH(6); EVICT/FAIL durations are time-to-failure, not runtime.
    completed = df[(df['terminal_type'] == 6) & (df['duration_sec'] > 0)]
    if len(completed) > 0:
        print(f'  Completed (FINISH): {len(completed)} — '
              f'duration median={completed["duration_sec"].median():.0f}s, '
              f'mean={completed["duration_sec"].mean():.0f}s')

print('\n✅ Job metadata done!')

---
## Dataset 5: Batch Job Statistical Profiles (Distribution Fitting)

This is the key methodological improvement over raw event replay.  
Following **Grange et al.** and **Da Costa et al. (2016)**, we fit statistical 
distributions to three batch job characteristics:

1. **Inter-arrival times** — time between consecutive batch job submissions
2. **Job durations** — wall-clock time from submit to completion
3. **Resource requests** — CPU and memory per task

For each, we fit candidate distributions (exponential, lognormal, gamma, Weibull)
and select the best fit by KS-test. The fitted parameters are exported for use 
in a synthetic workload generator.

### Why this approach?
- Allows controlled experiments: vary load intensity, job mix, flexibility factor
- Standard methodology in scheduling literature (reproducible)
- Avoids trace artifacts (cold-start, Borg-internal retries, alloc-set nesting)
- Produces a cleaner thesis methodology section

In [ ]:
def fit_best_distribution(data, name, candidates=None):
    """Fit candidate CONTINUOUS distributions; select the best by KS distance (D).

    The KS *p-value* is deliberately NOT used for selection or validation here:
      - With n ~ 1e5 it underflows to 0 for every candidate (KS power grows with
        sqrt(n)), so it cannot discriminate between fits.
      - Parameters are estimated from the same sample, so the textbook KS p-value
        is invalid anyway (this is the Lilliefors situation).
    We select the distribution that minimizes the KS D statistic and also report
    AIC so the fits can be compared on likelihood grounds.
    """
    if candidates is None:
        candidates = ['expon', 'lognorm', 'gamma', 'weibull_min']

    data = data[np.isfinite(data) & (data > 0)]
    if len(data) < 100:
        return {'distribution': 'insufficient_data', 'n_samples': len(data)}

    best, best_d = None, np.inf
    for dist_name in candidates:
        try:
            dist = getattr(stats, dist_name)
            params = dist.fit(data)
            ks_stat, ks_pval = stats.kstest(data, dist_name, args=params)
            loglik = float(np.sum(dist.logpdf(data, *params)))
            aic = 2 * len(params) - 2 * loglik
            if np.isfinite(ks_stat) and ks_stat < best_d:
                best_d = ks_stat
                best = {
                    'distribution': dist_name,
                    'params': [float(p) for p in params],
                    'ks_statistic': float(ks_stat),
                    'ks_pvalue': float(ks_pval),  # kept for reference; see docstring
                    'aic': float(aic) if np.isfinite(aic) else None,
                    'loglik': loglik if np.isfinite(loglik) else None,
                }
        except Exception:
            continue
    if best is None:
        best = {'distribution': 'fit_failed'}

    # Summary statistics for sanity checks (and as a non-parametric fallback)
    best.update({
        'name': name,
        'distribution_kind': 'continuous',
        'n_samples': len(data),
        'mean': float(np.mean(data)),
        'median': float(np.median(data)),
        'std': float(np.std(data)),
        'p5': float(np.percentile(data, 5)),
        'p25': float(np.percentile(data, 25)),
        'p75': float(np.percentile(data, 75)),
        'p95': float(np.percentile(data, 95)),
    })
    return best


def fit_best_discrete_distribution(data, name):
    """Fit DISCRETE count distributions (Poisson, geometric, negative-binomial)
    for count data such as tasks-per-job, where a continuous fit is inappropriate.

    scipy's discrete distributions have no .fit(), so parameters are estimated in
    closed form (MLE / method-of-moments) and scored with a discrete KS D statistic
    (max |empirical CDF - fitted CDF| over the integer support).
    """
    data = data[np.isfinite(data)]
    data = np.round(data).astype(int)
    data = data[data >= 0]
    if len(data) < 100:
        return {'distribution': 'insufficient_data', 'n_samples': len(data)}

    n = len(data)
    mean = float(data.mean())
    var = float(data.var())

    # Empirical CDF evaluated on the full integer support
    xs = np.arange(data.min(), data.max() + 1)
    sorted_data = np.sort(data)
    emp_cdf = np.searchsorted(sorted_data, xs, side='right') / n

    # Candidate (dist, params) pairs, each estimated in closed form
    candidates = {}
    if mean > 0:
        candidates['poisson'] = (stats.poisson, (mean,))            # MLE: lambda = mean
    if mean >= 1:
        candidates['geom'] = (stats.geom, (1.0 / mean,))            # MLE: p = 1/mean, support {1,2,...}
    if var > mean > 0:                                              # negative-binomial needs overdispersion
        p = mean / var
        r = mean * p / (1.0 - p)                                   # = mean^2 / (var - mean)
        if r > 0 and 0.0 < p < 1.0:
            candidates['nbinom'] = (stats.nbinom, (r, p))

    best, best_d = None, np.inf
    for dist_name, (dist, params) in candidates.items():
        try:
            cdf_vals = dist.cdf(xs, *params)
            ks_stat = float(np.max(np.abs(emp_cdf - cdf_vals)))
            loglik = float(np.sum(dist.logpmf(data, *params)))
            if np.isfinite(ks_stat) and ks_stat < best_d:
                best_d = ks_stat
                best = {
                    'distribution': dist_name,
                    'params': [float(x) for x in params],
                    'ks_statistic': ks_stat,
                    'aic': float(2 * len(params) - 2 * loglik) if np.isfinite(loglik) else None,
                    'loglik': loglik if np.isfinite(loglik) else None,
                }
        except Exception:
            continue
    if best is None:
        best = {'distribution': 'fit_failed'}

    best.update({
        'name': name,
        'distribution_kind': 'discrete',
        'n_samples': int(n),
        'mean': mean,
        'median': float(np.median(data)),
        'std': float(np.std(data)),
        'p5': float(np.percentile(data, 5)),
        'p25': float(np.percentile(data, 25)),
        'p75': float(np.percentile(data, 75)),
        'p95': float(np.percentile(data, 95)),
    })
    return best


all_cell_profiles = {}

for cell in CELLS:
    print(f'\n{"="*50}')
    print(f'Cell {cell}: Fitting batch job distributions')
    print(f'{"="*50}')

    # Load the job metadata we already extracted
    jobs = pd.read_csv(f'data/jobs/jobs_{cell}.csv')
    batch = jobs[(jobs['job_type'] == 'batch')].copy()
    # Durations only make sense for jobs that actually FINISHed (terminal_type == 6).
    completed_batch = batch[(batch['terminal_type'] == 6) & (batch['duration_sec'] > 0)].copy()

    print(f'  Total batch jobs: {len(batch)}')
    print(f'  Completed (FINISH) batch jobs: {len(completed_batch)}')

    if len(batch) < 100:
        print(f'  ⚠ Too few batch jobs, skipping distribution fitting')
        continue

    profiles = {}

    # 1. Inter-arrival times — use ALL batch submissions. The arrival process is
    #    not conditional on completion; filtering to completed jobs removes ~half
    #    the arrivals and artificially inflates the inter-arrival gaps.
    submit_times = np.sort(batch['submit_time'].values)
    inter_arrivals = np.diff(submit_times) / 1e6  # convert μs → seconds
    inter_arrivals = inter_arrivals[inter_arrivals > 0]
    fit = fit_best_distribution(inter_arrivals, 'inter_arrival_sec')
    profiles['inter_arrival'] = fit
    print(f'\n  Inter-arrival times:')
    print(f'    Best fit: {fit["distribution"]} (KS D={fit.get("ks_statistic", float("nan")):.4f})')
    print(f'    Mean={fit["mean"]:.1f}s, Median={fit["median"]:.1f}s')

    # 2. Job durations — only completed (FINISH) jobs have a meaningful runtime.
    durations = completed_batch['duration_sec'].values
    fit = fit_best_distribution(durations, 'duration_sec')
    profiles['duration'] = fit
    print(f'\n  Job durations:')
    print(f'    Best fit: {fit["distribution"]} (KS D={fit.get("ks_statistic", float("nan")):.4f})')
    print(f'    Mean={fit["mean"]:.0f}s, Median={fit["median"]:.0f}s')

    # 3-4. Resource requests are known at submit time, so use ALL batch jobs.
    cpu_req = batch['avg_cpu_request'].dropna().values
    fit = fit_best_distribution(cpu_req, 'cpu_request')
    profiles['cpu_request'] = fit
    print(f'\n  CPU request per task:')
    print(f'    Best fit: {fit["distribution"]} (KS D={fit.get("ks_statistic", float("nan")):.4f})')
    print(f'    Mean={fit["mean"]:.4f}, Median={fit["median"]:.4f}')

    mem_req = batch['avg_mem_request'].dropna().values
    fit = fit_best_distribution(mem_req, 'memory_request')
    profiles['memory_request'] = fit
    print(f'\n  Memory request per task:')
    print(f'    Best fit: {fit["distribution"]} (KS D={fit.get("ks_statistic", float("nan")):.4f})')
    print(f'    Mean={fit["mean"]:.4f}, Median={fit["median"]:.4f}')

    # 5. Tasks per job is DISCRETE count data → use the discrete fitter.
    tasks_per_job = batch['num_tasks'].dropna().values
    fit = fit_best_discrete_distribution(tasks_per_job, 'tasks_per_job')
    profiles['tasks_per_job'] = fit
    print(f'\n  Tasks per job (discrete):')
    print(f'    Best fit: {fit["distribution"]} (KS D={fit.get("ks_statistic", float("nan")):.4f})')
    print(f'    Mean={fit["mean"]:.1f}, Median={fit["median"]:.1f}')

    # 6. Workload mix ratio
    total_jobs = len(jobs)
    profiles['workload_mix'] = {
        'total_jobs': int(total_jobs),
        'batch_count': int((jobs['job_type'] == 'batch').sum()),
        'service_count': int((jobs['job_type'] == 'service').sum()),
        'batch_fraction': float((jobs['job_type'] == 'batch').mean()),
    }
    print(f'\n  Workload mix: {profiles["workload_mix"]["batch_fraction"]:.1%} batch')

    # Save per-cell profile
    with open(f'data/jobs/batch_distributions_{cell}.json', 'w') as f:
        json.dump(profiles, f, indent=2)
    all_cell_profiles[cell] = profiles

print('\n✅ Distribution fitting done!')

---
## Dataset 6: Unified Workload Generator Parameters

Aggregates the per-cell distribution fits into a single parameter file
that can drive a synthetic workload generator. Also includes the 
flexibility-factor methodology from Grange et al. for generating deadlines:

```
deadline = submit_time + duration × (1 + flexibility_factor)
```

Where `flexibility_factor ∈ [0.5, 1.0, 2.0, 4.0]` controls how much slack
batch jobs have — this is a key experimental variable in Grange et al.

In [ ]:
# Pick the primary cell data-drivenly: the cell with the most completed (FINISH)
# batch jobs — i.e. the largest sample for the batch distributions this generator
# produces. (Cell 'a' was NOT a good default: it has the fewest batch jobs and the
# lowest batch fraction of the four cells.)
# NOTE: the cells differ substantially (see cross_validation below: batch fraction
# ranges ~4%–48%, mean duration ~2000–5700s). For the final experiments, consider
# pooling all cells or treating each cell as a separate scenario rather than relying
# on a single "primary" cell.
primary_cell = (
    max(all_cell_profiles, key=lambda c: all_cell_profiles[c]['duration'].get('n_samples', 0))
    if all_cell_profiles else None
)

if primary_cell is not None:
    p = all_cell_profiles[primary_cell]

    generator_params = {
        'description': (
            'Workload generator parameters fitted to Google ClusterData2019. '
            'Distributions fitted via MLE, selected by minimum KS D statistic. '
            'Following Da Costa et al. (2016) and Grange et al. (2018) methodology.'
        ),
        'primary_cell': primary_cell,
        'primary_cell_selection': 'cell with most completed (FINISH) batch jobs',
        'inter_arrival': p['inter_arrival'],
        'duration': p['duration'],
        'cpu_request': p['cpu_request'],
        'memory_request': p['memory_request'],
        'tasks_per_job': p['tasks_per_job'],
        'workload_mix': p['workload_mix'],
        'deadline_model': {
            'description': 'deadline = submit_time + duration * (1 + flexibility_factor)',
            'flexibility_factors': [0.5, 1.0, 2.0, 4.0],
            'note': 'Grange et al. sweep these values; higher = more scheduling slack'
        },
        'cross_validation': {
            cell: {
                'batch_fraction': all_cell_profiles[cell]['workload_mix']['batch_fraction'],
                'mean_duration': all_cell_profiles[cell]['duration']['mean'],
                'mean_inter_arrival': all_cell_profiles[cell]['inter_arrival']['mean'],
                'n_completed_batch': all_cell_profiles[cell]['duration'].get('n_samples', 0),
            }
            for cell in all_cell_profiles
        }
    }

    with open('data/workload_generator_params.json', 'w') as f:
        json.dump(generator_params, f, indent=2)
    print('Saved data/workload_generator_params.json')

    print(f'\nPrimary cell ({primary_cell}) generator parameters:')
    print(f'  Inter-arrival: {p["inter_arrival"]["distribution"]} '
          f'(mean={p["inter_arrival"]["mean"]:.1f}s)')
    print(f'  Duration:      {p["duration"]["distribution"]} '
          f'(mean={p["duration"]["mean"]:.0f}s)')
    print(f'  CPU request:   {p["cpu_request"]["distribution"]} '
          f'(mean={p["cpu_request"]["mean"]:.4f})')
    print(f'  Tasks/job:     {p["tasks_per_job"]["distribution"]} '
          f'(mean={p["tasks_per_job"]["mean"]:.1f})')
    print(f'  Batch fraction: {p["workload_mix"]["batch_fraction"]:.1%}')

    print(f'\nCross-cell validation:')
    for cell in all_cell_profiles:
        cv = generator_params['cross_validation'][cell]
        print(f'  Cell {cell}: batch={cv["batch_fraction"]:.1%}, '
              f'dur_mean={cv["mean_duration"]:.0f}s, '
              f'iat_mean={cv["mean_inter_arrival"]:.1f}s, '
              f'n_completed={cv["n_completed_batch"]}')
else:
    print('⚠ No cell profiles available — check Dataset 4 and 5 outputs')

print('\n✅ Workload generator params done!')

---
## Validation Plots

Visual sanity checks: distribution fits, utilization curves, machine heterogeneity.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('ClusterData2019 Extraction Summary', fontsize=14, fontweight='bold')

# 1. Cell utilization (cell a)
ax = axes[0, 0]
cell_df = pd.read_csv('data/cells/cell_a.csv')
# Downsample for plotting
step = max(1, len(cell_df) // 2000)
ax.plot(cell_df['timestep'].values[::step] / 12 / 24,
        cell_df['cpu_demand_norm'].values[::step], linewidth=0.5, alpha=0.8)
ax.set_xlabel('Days')
ax.set_ylabel('CPU Utilization')
ax.set_title('Cell A: CPU Demand (5-min)')
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

# 2. Power model
ax = axes[0, 1]
scatter = pd.read_csv('data/power_model_scatter.csv')
ax.scatter(scatter['cpu_util'], scatter['power_util'], alpha=0.05, s=1)
with open('data/power_model_params.json') as f:
    pm = json.load(f)
x_line = np.linspace(0, scatter['cpu_util'].max(), 100)
ax.plot(x_line, pm['idle_power'] + pm['slope'] * x_line, 'r-', linewidth=2,
        label=f'P={pm["idle_power"]:.3f}+{pm["slope"]:.3f}×CPU\nR²={pm["r_squared"]:.3f}')
ax.set_xlabel('CPU Utilization')
ax.set_ylabel('Power Utilization')
ax.set_title('Power Model Fit')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 3. Machine heterogeneity
ax = axes[0, 2]
machines = pd.read_csv('data/machines/machines_all.csv')
ax.scatter(machines['cpu_capacity'], machines['memory_capacity'],
           alpha=0.1, s=2, c=machines['cell'].map({'a':0,'b':1,'c':2,'d':3}), cmap='tab10')
ax.set_xlabel('CPU Capacity (normalized)')
ax.set_ylabel('Memory Capacity (normalized)')
ax.set_title(f'Machine Fleet ({len(machines)} servers)')
ax.grid(True, alpha=0.3)

# 4. Job duration distribution (batch, cell a) — FINISH-only, matching the fit
ax = axes[1, 0]
jobs_a = pd.read_csv('data/jobs/jobs_a.csv')
batch_dur = jobs_a[(jobs_a['job_type']=='batch') &
                   (jobs_a['terminal_type'] == 6) &
                   (jobs_a['duration_sec'] > 0)]['duration_sec']
if len(batch_dur) > 0:
    ax.hist(np.log10(batch_dur.clip(lower=1)), bins=80, density=True, alpha=0.7, color='steelblue')
    ax.set_xlabel('log₁₀(Duration / seconds)')
    ax.set_ylabel('Density')
ax.set_title(f'Batch Job Durations (Cell A, n={len(batch_dur)})')
ax.grid(True, alpha=0.3)

# 5. Workload mix across cells
ax = axes[1, 1]
mix_data = []
for cell in CELLS:
    jf = pd.read_csv(f'data/jobs/jobs_{cell}.csv')
    mix_data.append({
        'cell': cell,
        'batch': (jf['job_type']=='batch').sum(),
        'service': (jf['job_type']=='service').sum()
    })
mix_df = pd.DataFrame(mix_data)
x = range(len(mix_df))
ax.bar(x, mix_df['batch'], label='Batch', color='steelblue')
ax.bar(x, mix_df['service'], bottom=mix_df['batch'], label='Service', color='coral')
ax.set_xticks(x)
ax.set_xticklabels([f'Cell {c}' for c in CELLS])
ax.set_ylabel('Number of Jobs')
ax.set_title('Workload Mix by Cell')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# 6. Inter-arrival time distribution (cell a) — all batch submissions, matching the fit
ax = axes[1, 2]
batch_a = jobs_a[(jobs_a['job_type']=='batch') & jobs_a['submit_time'].notna()].sort_values('submit_time')
iat = np.diff(batch_a['submit_time'].values) / 1e6  # μs → seconds
iat = iat[(iat > 0) & (iat < np.percentile(iat[iat>0], 99))]
if len(iat) > 0:
    ax.hist(iat, bins=100, density=True, alpha=0.7, color='steelblue')
    ax.set_xlabel('Inter-arrival Time (seconds)')
    ax.set_ylabel('Density')
ax.set_title(f'Batch Inter-arrival (Cell A, n={len(iat)})')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/extraction_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved data/extraction_summary.png')

---
## Summary & Download

In [ ]:
print('=' * 65)
print('EXTRACTION SUMMARY')
print('=' * 65)

for cell in CELLS:
    print(f'\nCell {cell}:')
    for label, path in [
        ('Utilization (5min)', f'data/cells/cell_{cell}.csv'),
        ('Machines',           f'data/machines/machines_{cell}.csv'),
        ('Jobs (all)',         f'data/jobs/jobs_{cell}.csv'),
        ('Batch distributions', f'data/jobs/batch_distributions_{cell}.json'),
    ]:
        if os.path.exists(path):
            if path.endswith('.csv'):
                n = len(pd.read_csv(path))
                print(f'  {label:25s} {n:>10,} rows')
            else:
                size = os.path.getsize(path)
                print(f'  {label:25s} {size/1024:>10.1f} KB')

print(f'\nGlobal files:')
for label, path in [
    ('Power model',           'data/power_model_params.json'),
    ('All machines',          'data/machines/machines_all.csv'),
    ('Generator params',      'data/workload_generator_params.json'),
    ('Power scatter',         'data/power_model_scatter.csv'),
    ('Summary plot',          'data/extraction_summary.png'),
]:
    if os.path.exists(path):
        if path.endswith('.csv'):
            n = len(pd.read_csv(path))
            print(f'  {label:25s} {n:>10,} rows')
        else:
            size = os.path.getsize(path)
            print(f'  {label:25s} {size/1024:>10.1f} KB')

print('\n' + '=' * 65)
print('BENCHMARK PAPER MAPPING')
print('=' * 65)
print("""
Grange et al. (2018) — Batch scheduling + renewable awareness:
  → workload_generator_params.json  (generate synthetic batch jobs)
  → cells/cell_*.csv               (aggregate demand for capacity)
  → power_model_params.json        (energy cost model)
  → YOU ADD: solar trace from NREL/PVGIS + electricity prices from ComEd/IESO
  → Deadline formula: submit_time + duration × (1 + flexibility_factor)

Xu et al. (2020) — Self-adaptive brownout + batch deferral:
  → jobs/jobs_*.csv                (batch/service classification + resource requests)
  → cells/cell_*.csv               (utilization time series for brownout triggers)
  → power_model_params.json        (energy model)
  → YOU ADD: renewable energy trace + brown/green energy pricing

Haghshenas et al. (2022) — Infrastructure-aware heterogeneous scheduling:
  → machines/machines_all.csv      (heterogeneous server fleet)
  → jobs/jobs_*.csv                (heterogeneous workload mix)
  → cells/cell_*.csv               (cooling model input — utilization drives heat)
  → power_model_params.json        (per-server-type power model)
  → YOU ADD: cooling power model + tiered electricity rate structure
""")

In [ ]:
# Download everything as zip
import shutil
from google.colab import files

shutil.make_archive('clusterdata2019_full', 'zip', '.', 'data')
files.download('clusterdata2019_full.zip')
print('Download started!')
print('Extract into C:\\Projects\\thesis\\ on your local machine.')